Mounted at /content/drive


In [ ]:
# Unzip the archive into a local 'data' folder
!unzip -q "/content/drive/My Drive/Alzheimer_Project/archive.zip" -d "/content/data"

# This command prints the contents to verify the extraction worked
import os
print("Folders found:", os.listdir("/content/data"))

replace /content/data/Data/Mild Dementia/OAS1_0028_MR1_mpr-1_100.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

# 1. THE GYM: This randomly "messes" with the images during training
# so the AI learns the actual brain shape, not just the pixels.
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

# 2. THE BRAIN: Use MobileNetV2 but freeze the "base" knowledge
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

# 3. THE SKELETON: Combine everything
model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    data_augmentation,          # <--- ADDED THIS HERE
    layers.Rescaling(1./255),   # Standardizes pixels to [0,1] automatically
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),        # Increased to 0.4 to prevent "stubborn" guessing
    layers.Dense(4, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print("Pro Model with Data Augmentation is ready!")

In [ ]:
# Save in the new, universal Keras format
model.save('/content/alzheimer_model.keras')
print("Universal model saved!")

Universal model saved!


In [ ]:
# This saves ONLY the numerical weights, no configuration metadata
model.save_weights('/content/alzheimer_weights.weights.h5')

In [ ]:
import tensorflow as tf

# 1. SET THE PATH (Make sure this matches your 'Data' folder)
DATA_PATH = '/content/data/Data'

# 2. RE-LOAD THE DATASETS
print("--- Re-loading Dataset ---")
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_PATH,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_PATH,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

print("Datasets are back in memory!")

--- Re-loading Dataset ---
Found 86437 files belonging to 4 classes.
Using 69150 files for training.
Found 86437 files belonging to 4 classes.
Using 17287 files for validation.
Datasets are back in memory!


In [ ]:
# 1. SETUP CLASS WEIGHTS
# This forces the AI to pay 5x more attention to 'Mild' and 2x more to 'Non Demented'
# It stops the AI from being "lazy" and just guessing 'Very Mild' every time.
class_weight = {
    0: 2.0,  # Mild Dementia
    1: 5.0,  # Moderate Dementia (Rare, so we weight it high)
    2: 1.0,  # Non Demented
    3: 1.0   # Very Mild Dementia
}

# 2. THE TRAINING COMMAND
print("--- Starting Pro Training with Augmentation ---")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weight
)

# 3. SAVE THE NEW WEIGHTS
model.save_weights('/content/alzheimer_pro_weights.weights.h5')
print("DONE! New weights saved as alzheimer_pro_weights.weights.h5")

--- Starting Pro Training with Augmentation ---
Epoch 1/10
2161/2161 ━━━━━━━━━━━━━━━━━━━━ 171s 74ms/step - accuracy: 0.7632 - loss: 0.7962 - val_accuracy: 0.7942 - val_loss: 0.5032
Epoch 2/10
2161/2161 ━━━━━━━━━━━━━━━━━━━━ 171s 79ms/step - accuracy: 0.7719 - loss: 0.7174 - val_accuracy: 0.7896 - val_loss: 0.5433
Epoch 3/10
2161/2161 ━━━━━━━━━━━━━━━━━━━━ 189s 73ms/step - accuracy: 0.7735 - loss: 0.7135 - val_accuracy: 0.8015 - val_loss: 0.5018
Epoch 4/10
2161/2161 ━━━━━━━━━━━━━━━━━━━━ 158s 73ms/step - accuracy: 0.7744 - loss: 0.7084 - val_accuracy: 0.7994 - val_loss: 0.4929
Epoch 5/10
2161/2161 ━━━━━━━━━━━━━━━━━━━━ 158s 73ms/step - accuracy: 0.7734 - loss: 0.7019 - val_accuracy: 0.8008 - val_loss: 0.4872
Epoch 6/10
2161/2161 ━━━━━━━━━━━━━━━━━━━━ 202s 73ms/step - accuracy: 0.7740 - loss: 0.7034 - val_accuracy: 0.8021 - val_loss: 0.4863
Epoch 7/10
2161/2161 ━━━━━━━━━━━━━━━━━━━━ 170s 78ms/step - accuracy: 0.7752 - loss: 0.6961 - val_accuracy: 0.8089 - val_loss: 0.4704
Epoch 8/10
2161/2161 

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="oKfwa7JvH0bhB5nMQqCE")
project = rf.workspace("deep-learning-xe457").project("alzheimer-s-disease-detection-ud5st")
version = project.version(1)
dataset = version.download("tfrecord")


loading Roboflow workspace...
loading Roboflow project...


In [ ]:
import tensorflow as tf

def _parse_roboflow_data(example_proto):
    # This structure is standard for Roboflow TFRecords
    feature_description = {
        'image/encoded': tf.io.FixedLenFeature([], tf.string),
        'image/class/label': tf.io.FixedLenFeature([], tf.int64),
    }
    features = tf.io.parse_single_example(example_proto, feature_description)
    image = tf.image.decode_jpeg(features['image/encoded'], channels=3)
    image = tf.image.resize(image, [224, 224])
    label = tf.cast(features['image/class/label'], tf.int32)
    return image, label

# Ensure this path is inside the quotes and nothing follows it
tfrecord_path = "/content/roboflow_data/train/Alzheimers.tfrecord"

roboflow_ds = tf.data.TFRecordDataset(tfrecord_path)
roboflow_ds = roboflow_ds.map(_parse_roboflow_data).batch(32)

print("✅ Roboflow data is decoded and ready for the Super Dataset!")

✅ Roboflow data is decoded and ready for the Super Dataset!


In [ ]:
# 'train_ds' is your original OASIS data
# 'roboflow_ds' is your new data
combined_train_ds = train_ds.concatenate(roboflow_ds).shuffle(buffer_size=1000)

print(f"🔥 Super Dataset created! Ready to fix those 'Non Demented' errors.")

🔥 Super Dataset created! Ready to fix those 'Non Demented' errors.


In [ ]:
# 1. Update the weights to be extra aggressive for the rare 'Moderate' class
# (Index 1 is usually Moderate Dementia)
super_class_weights = {0: 3.0, 1: 10.0, 2: 1.0, 3: 2.0}

# 2. Start the Training
print("--- Training the Super Model ---")
history = model.fit(
    combined_train_ds,      # Using your merged data!
    validation_data=val_ds, # Still validating against the clean split
    epochs=10,
    class_weight=super_class_weights
)

# 3. Save the "Golden" Weights
model.save_weights('/content/alzheimer_ultimate_model.weights.h5')
print("✅ SUCCESS! Download 'alzheimer_ultimate_model.weights.h5' for your Flask app.")

--- Training the Super Model ---


NameError: name 'model' is not defined